# Simple: Record Suppression with PAMOLA.CORE

**Goal**: Learn record suppression basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply condition-based record removal using execute()
- Compare before/after results
- Understand privacy protection through record removal

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/suppression/
        └── 01_record_suppression_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.record_op import (
        RecordSuppressionOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load from `examples/data_examples/sample.csv`
- Auto-creates 50 patient records if file missing
- Review dataset structure and identify null values

**What you'll see:**
- Dataset summary (50 records, 7 columns)
- First 5 records with patient data
- Column statistics (types, unique counts, null counts)
- Some records with null diagnosis/zipcode for demo

**Note:** Auto-generated data includes realistic null values for suppression scenarios

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with various suppression scenarios
    np.random.seed(42)
    sample_data = pd.DataFrame({
        'record_id': range(1, 51),
        'patient_id': [f'P{i:04d}' for i in range(1, 51)],
        'age': np.random.randint(18, 85, 50),
        'diagnosis': np.random.choice(
            ['Diabetes', 'Hypertension', 'Asthma', 'None', None],
            50,
            p=[0.3, 0.3, 0.2, 0.15, 0.05]
        ),
        'risk_score': np.random.randint(1, 11, 50),
        'zipcode': np.random.choice(
            ['10001', '10002', '10003', '10004', '10005', None],
            50,
            p=[0.2, 0.2, 0.2, 0.2, 0.15, 0.05]
        ),
        'treatment_cost': np.random.randint(500, 5000, 50)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        null_count = df[col].isna().sum()
        unique_count = df[col].nunique()
        print(f"  {col:20s} ({dtype_str:10s}): {unique_count} unique, {null_count} nulls")
    else:
        null_count = df[col].isna().sum()
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}, {null_count} nulls")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name**: Edit `create_config_kwargs()` function
   - Default: `"email"`
   - Change to YOUR target field (e.g., "diagnosis")
2. Run to validate and setup environment

**What you'll see:**
- Field validation (unique count, null count, sample values)
- Value distribution (top frequencies)
- Task directory, TaskReporter, DataSource initialized (✓)

**Note:** Records with null values in this field will be suppressed

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Field to check for suppression - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Null count: {df[field_name].isna().sum()}")
print(f"  Sample values: {list(df[field_name].dropna().unique()[:5])}")
print(f"  Distribution:")
print(df[field_name].value_counts(dropna=False).head())

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="record_suppression",
    description="Record suppression based on null values in 'diagnosis' field.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration summary
- Run to execute record suppression
- Monitor progress bar (6 steps: validate → check → remove → save → metrics → viz)

**Key parameters:**
- `suppression_mode='REMOVE'`: Remove entire records (required)
- `suppression_condition='null'`: Remove records with null values
- `save_suppressed_records=True`: Save removed records for audit

**What you'll see:**
- Configuration summary with all parameters
- Progress bar through 6 processing steps
- Completion status with artifact count (5-7 files)
- Success message (✅ Operation completed!)

**Note:** Execution takes ~1-3 seconds for 50 records

In [ ]:
# Create operation with production-style configuration
operation = RecordSuppressionOperation(
    # Core parameters
    field_name=field_name,               # From config
    suppression_mode='REMOVE',           # Must be 'REMOVE' for record suppression
    
    # Suppression condition parameters
    suppression_condition='null',        # Remove records with null values
    suppression_values=None,             # Not used for 'null' condition
    suppression_range=None,              # Not used for 'null' condition
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Suppressed records handling
    save_suppressed_records=True,        # Save removed records to separate file
    suppression_reason_field='_suppression_reason',  # Field name for reason
    
    # Privacy parameters (for risk-based suppression)
    ka_risk_field=None,                  # Not used for 'null' condition
    risk_threshold=5,                    # Not used for 'null' condition
    
    # Multi-condition parameters (for custom suppression)
    multi_conditions=None,               # Not used for 'null' condition
    condition_logic='AND',               # Not used for 'null' condition
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:                    {operation.field_name}")
print(f"  Suppression mode:         {operation.suppression_mode}")
print(f"  Suppression condition:    {operation.suppression_condition}")
print(f"  Save suppressed records:  {operation.save_suppressed_records}")
print(f"  Visualizations:           {operation.generate_visualization}")
print(f"  Save output:              {operation.save_output}")
print(f"  Force recalc:             {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing record suppression...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare processed data
- Review suppression statistics

**What you'll see:**
- Artifacts list (CSV, JSON, PNG files)
- Sample records (first 10) from remaining data
- Null count comparison (original vs processed)
- Suppression summary (original, remaining, suppressed counts, rate %)
- Location of suppressed records file

**Note:** Suppressed records saved separately for audit/compliance purposes

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
# Filter out suppressed records files
output_files = [f for f in output_files if 'suppressed' not in f.name]

if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    display_cols = [field_name]
    if 'patient_id' in result_df.columns:
        display_cols.append('patient_id')
    if 'age' in result_df.columns:
        display_cols.append('age')
    if 'risk_score' in result_df.columns:
        display_cols.append('risk_score')
    print(result_df[display_cols].head(10))
    
    # Compare field null counts
    print(f"\n📈 {field_name.title()} Null Count Comparison:")
    original_nulls = df[field_name].isna().sum()
    processed_nulls = result_df[field_name].isna().sum() if field_name in result_df.columns else 0
    print(f"  Original nulls:  {original_nulls} ({original_nulls/len(df)*100:.1f}%)")
    print(f"  Processed nulls: {processed_nulls} ({processed_nulls/len(result_df)*100:.1f}% if field exists)")
    
    # Calculate suppression metrics
    original_count = len(df)
    remaining_count = len(result_df)
    suppressed_count = original_count - remaining_count
    suppression_rate = (suppressed_count / original_count) * 100
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:    {original_count}")
    print(f"  Remaining records:   {remaining_count}")
    print(f"  Suppressed records:  {suppressed_count}")
    print(f"  Suppression rate:    {suppression_rate:.1f}%")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Records with null {field_name} were removed to improve data quality!")
    
    # Check for suppressed records file
    suppressed_files = list(task_dir.glob('output/suppressed_records/*.csv'))
    if suppressed_files:
        print(f"\n📁 Suppressed records saved to: {suppressed_files[0].parent}")
        print(f"   File count: {len(suppressed_files)}")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-3 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/                      # Processed CSV
│   └── suppressed_records/      # Removed records
├── metrics/                     # Suppression statistics JSON
├── visualizations/              # Before/after charts
└── config/                      # Operation configuration
```

**Note:** Suppressed records in separate folder for audit trail

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'output/suppressed_records', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        # Filter out directories for file count
        files = [f for f in files if f.is_file()]
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points in output

**What you'll see:**
1. **Metrics JSON**: Suppression condition, records suppressed, rate, breakdown
2. **Output Data**: First 10 remaining records, field statistics, distribution
3. **Suppressed Records**: Removed records with suppression reasons
4. **Visualizations**: Before/after comparison charts

**Validate:**
- ✅ Records with null values removed
- ✅ Suppression rate matches expectations
- ✅ Remaining data has no nulls in target field
- ✅ Suppressed records properly saved for audit

**Note:** Only newest files shown. Multiple runs create versions - older artifacts excluded automatically

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]
        target_files = filtered if filtered else metrics_files

        if filtered:
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # Pick newest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # -----------------------------
            # 1️⃣ METADATA
            # -----------------------------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            op = metadata.get("operation", {})
            print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # -----------------------------
            # 2️⃣ HIGH-LEVEL SUPPRESSION METRICS
            # -----------------------------
            print("\n📊 SUPPRESSION METRICS:")
            hl_keys = [
                "operation_type",
                "suppression_condition",
                "records_suppressed",
                "suppression_rate",
                "remaining_records",
            ]
            for k in hl_keys:
                if k in metrics:
                    print(f"   {k.replace('_', ' ').title()}: {metrics[k]}")

            if "suppression_by_condition" in metrics:
                print("   Breakdown:")
                for cond, cnt in metrics["suppression_by_condition"].items():
                    print(f"      {cond}: {cnt}")

            # -----------------------------
            # 3️⃣ FIELD-LEVEL EFFECTIVENESS METRICS
            # Auto-group by prefix: resume_id, phone, skills, ...
            # -----------------------------
            print("\n📈 FIELD EFFECTIVENESS METRICS:")

            groups = {}

            for k, v in metrics.items():
                if k.endswith(("_effectiveness_ratio",
                               "_original_unique",
                               "_anonymized_unique",
                               "_null_increase")):

                    field = k.rsplit("_", 3)[0]  # remove last suffix
                    groups.setdefault(field, {})
                    groups[field][k] = v

            # sort by field name
            for field in sorted(groups.keys()):
                g = groups[field]

                print(f"\n   ▶ {field}")
                if f"{field}_effectiveness_ratio" in g:
                    print(f"      Effectiveness: {g[f'{field}_effectiveness_ratio']:.4f}")
                if f"{field}_original_unique" in g:
                    print(f"      Original unique: {g[f'{field}_original_unique']}")
                if f"{field}_anonymized_unique" in g:
                    print(f"      Anonymized unique: {g[f'{field}_anonymized_unique']}")
                if f"{field}_null_increase" in g:
                    print(f"      Null increase: {g[f'{field}_null_increase']:.4f}")

            # -----------------------------
            # 4️⃣ PERFORMANCE
            # -----------------------------
            print("\n⚡ PERFORMANCE:")
            perf_keys = [
                "duration_seconds",
                "records_processed",
                "records_per_second",
                "chunk_count",
                "chunk_size_used",
                "adaptive_chunk_size",
            ]
            for k in perf_keys:
                if k in metrics:
                    print(f"   {k.replace('_', ' ').title()}: {metrics[k]}")

            # -----------------------------
            # 5️⃣ EFFECTIVENESS SUMMARY
            # -----------------------------
            print("\n📘 EFFECTIVENESS SUMMARY:")
            summary_keys = [
                "total_records",
                "processed_records",
                "filtered_records",
                "processing_rate",
            ]
            for k in summary_keys:
                if k in metrics:
                    print(f"   {k.replace('_', ' ').title()}: {metrics[k]}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Sample (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    # Filter out suppressed records
    output_files = [f for f in output_files if 'suppressed' not in f.name]
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show field statistics
            if field_name in output_df.columns:
                print(f"\n\n📊 {field_name.title()} Statistics After Suppression:")
                print("-" * 80)
                print(f"  Null count: {output_df[field_name].isna().sum()}")
                print(f"  Unique values: {output_df[field_name].nunique()}")
                print(f"\n  Value Distribution:")
                dist = output_df[field_name].value_counts(dropna=False).head(10)
                for val, count in dist.items():
                    pct = (count / len(output_df)) * 100
                    print(f"     {str(val):<30}: {count:>5} ({pct:>5.1f}%)")
            else:
                print(f"\n⚠️  Field '{field_name}' not found in output")
                print(f"Available columns: {', '.join(output_df.columns)}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Suppressed Records (NEWEST FILE)
print("\n\n3️⃣ SUPPRESSED RECORDS:")
print("-" * 80)

suppressed_dir = task_dir / 'output' / 'suppressed_records'
if not suppressed_dir.exists():
    print(f"⚠️  Suppressed records directory not found: {suppressed_dir}")
else:
    suppressed_files = list(suppressed_dir.glob('*.csv'))
    
    if not suppressed_files:
        print("⚠️  No suppressed record files found")
    else:
        # Sort by modification time (newest first)
        suppressed_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_suppressed_file = suppressed_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_suppressed_file.stat().st_mtime)
        print(f"✓ Found {len(suppressed_files)} suppressed record file(s)")
        print(f"📄 Loading LATEST: {latest_suppressed_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_suppressed_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            suppressed_df = pd.read_csv(latest_suppressed_file)
            print(f"📊 Shape: {suppressed_df.shape[0]} rows × {suppressed_df.shape[1]} columns")
            print(f"\n{suppressed_df.head(10).to_string()}")
            
            # Show suppression reason if available
            if '_suppression_reason' in suppressed_df.columns:
                print(f"\n\n📋 Suppression Reasons:")
                print("-" * 80)
                reason_dist = suppressed_df['_suppression_reason'].value_counts()
                for reason, count in reason_dist.items():
                    print(f"   {reason}: {count} records")
        
        except Exception as e:
            print(f"❌ Error reading suppressed records: {e}")

# 4. Display Visualizations (NEWEST FILES ONLY)
print("\n\n4️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                name_parts = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {name_parts}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review metrics, output data, and suppressed records

**Key takeaways:**
- `operation.execute()` handles end-to-end processing
- Execution behavior (visualizations, output saving, caching) configured in operation constructor
- Field name is configurable via `create_config_kwargs()`
- `suppression_condition='null'` removes records with missing values
- `save_suppressed_records=True` preserves removed records for audit trails
- Suppression improves data quality by removing incomplete records
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_record_suppression_advanced.ipynb`** includes:
- All 5 suppression conditions (null, value, range, risk, custom)
- Multi-field custom conditions with AND/OR logic
- K-anonymity risk-based suppression
- Batch processing with parallel execution
- Advanced audit trails and compliance tracking
- Production deployment patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Privacy Metrics Guide](../../docs/privacy_metrics.md)
- 📊 [Suppression Strategies Guide](../../docs/suppression_strategies.md)